In [15]:
%pip -q install requests lxml cssselect


[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: python3 -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [17]:
import requests
from lxml import html
from urllib.parse import urljoin, unquote

def scrape_quests_api():
    """Use Fandom's MediaWiki API to get quest page content"""
    api_url = "https://arcanum.fandom.com/api.php"
    
    params = {
        "action": "parse",
        "page": "Quests",
        "prop": "text",
        "format": "json"
    }
    
    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36"
    }
    
    try:
        response = requests.get(api_url, params=params, headers=headers)
        response.raise_for_status()
        data = response.json()
        
        html_content = data['parse']['text']['*']
        tree = html.fromstring(html_content)
        
        # Skill quests: td:last-child > a
        skill_elements = tree.cssselect('td:last-child > a')
        skill_urls = []
        for elem in skill_elements:
            href = elem.get('href')
            if href and '/wiki/' in href:
                full_url = urljoin("https://arcanum.fandom.com", href.split('?')[0])
                skill_urls.append(full_url)
        
        # Other quests: //li//a[@href[starts-with(.,"/wiki")]]
        other_elements = tree.xpath('//li//a[@href[starts-with(.,"/wiki")]]')
        other_urls = []
        for elem in other_elements:
            href = elem.get('href')
            if href:
                full_url = urljoin("https://arcanum.fandom.com", href.split('?')[0])
                if full_url not in skill_urls:
                    other_urls.append(full_url)
        
        # Remove duplicates
        seen = set()
        skill_urls = [x for x in skill_urls if not (x in seen or seen.add(x))]
        other_urls = [x for x in other_urls if not (x in seen or seen.add(x))]
        
        return {
            'skill_quests': skill_urls,
            'other_quests': other_urls,
            'total': len(skill_urls) + len(other_urls)
        }
        
    except Exception as e:
        print(f"Error: {e}")
        return None


In [18]:
result = scrape_quests_api()

if result:
    print(f"=== Skill Quests ({len(result['skill_quests'])}) ===")
    for url in result['skill_quests']:
        print(url)
        
    print(f"\n=== Other Quests ({len(result['other_quests'])}) ===")
    for url in result['other_quests']:
        print(url)

=== Skill Quests (16) ===
https://arcanum.fandom.com/wiki/Find_the_Bow_of_Ecclesiastes
https://arcanum.fandom.com/wiki/Kill_Sir_Garrick_Stout
https://arcanum.fandom.com/wiki/Find_Lady_Druella
https://arcanum.fandom.com/wiki/Retrieve_Azram%27s_Star
https://arcanum.fandom.com/wiki/Find_the_Master_of_Backstab
https://arcanum.fandom.com/wiki/Run_Around_Tarant_in_Your_Underwear
https://arcanum.fandom.com/wiki/Find_the_Master_of_Prowling
https://arcanum.fandom.com/wiki/Get_staff_of_K%E2%80%99an_T%E2%80%99au
https://arcanum.fandom.com/wiki/Gamble_with_Gurin_Rockharrow
https://arcanum.fandom.com/wiki/Acquire_Ten_Thousand_Gold_Pieces
https://arcanum.fandom.com/wiki/Find_the_Master_of_Heal
https://arcanum.fandom.com/wiki/Negotiations_with_Caladon
https://arcanum.fandom.com/wiki/Find_Proof_that_Maxim%27s_Air_Machines_Flew
https://arcanum.fandom.com/wiki/Rescue_Mrs._Rolland_Unharmed
https://arcanum.fandom.com/wiki/Free_J.T._Morgan
https://arcanum.fandom.com/wiki/Survive_the_Training_Maze

=== Othe